# N175 · 陌生题迁移与结业项目

**目标：** 在无标签新变体中完成建模、证明、实现和复盘。

**先修：** N168, N169, N170, N171, N172, N173, N174, N108, N059。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 独立分析；强参照；约束变化；复杂度；解释；失败边界；知识迁移。

## 从问题到算法

结业不是再做一遍已展示的例题，而是迁移已有结构。先复习两条主线：带权区间调度用“选/不选当前工作＋二分前驱”；有序矩阵行和用“逐行合并有序和列表＋Top-k 截断”。重复和值不能去重，因为不同选择组合各占一个排名。

**迁移练习契约：** 输入 `instance={jobs:[(start,end,profit),...], max_jobs:K, cooldown:g}`；选择至多 K 份工作，前一份的 end+g 不大于后一份 start，允许一份不选，返回最大收益。相邻相接在 g=0 时合法；收益可负，因此最优值至少为零。

状态新增“还能选多少份”，而不是另造一个庞大框架。前驱条件从 end≤start 改成 end≤start−g。课程演示数据与测试随机源分离；这里的“陌生”指未展示给学习者的组合变体，不声称它是研究界全新问题或模型训练中从未出现过。

**结业提交：** 先写自己的推导与实现，再打开参考实现。评分分别记录接口正确、证明完整、边界处理、复杂度及约束改变后的迁移解释；不以已刷题数代替能力。

## 最小实现

**本章接口：** `job_scheduling(start, end, profit)`、`kth_smallest_row_sum(mat, k)`、`solve_unseen_variant(instance)`、`reference_unseen_variant(instance)`

In [1]:
from bisect import bisect_right
from itertools import combinations
import heapq

def job_scheduling(start,end,profit):
    if not len(start)==len(end)==len(profit):raise ValueError('Length mismatch')
    jobs=sorted(zip(end,start,profit))
    if any(left>=right for right,left,value in jobs):raise ValueError('Positive job duration required')
    ends=[right for right,left,value in jobs]
    dp=[0]*(len(jobs)+1)
    for i,(right,left,value) in enumerate(jobs,1):
        previous=bisect_right(ends,left,hi=i-1)
        dp[i]=max(dp[i-1],dp[previous]+value)
    return dp[-1]

def kth_smallest_row_sum(mat,k):
    if not mat or any(not row for row in mat):raise ValueError('Nonempty rows required')
    if k<1:raise ValueError('Positive rank required')
    combinations_count=1
    for row in mat:
        if any(a>b for a,b in zip(row,row[1:])):raise ValueError('Each row must be sorted')
        combinations_count*=len(row)
    if k>combinations_count:raise ValueError('Rank exceeds the number of choices')
    sums=[0]
    for row in mat:
        heap=[(value+row[0],i,0) for i,value in enumerate(sums)]
        heapq.heapify(heap);merged=[]
        while heap and len(merged)<k:
            value,i,j=heapq.heappop(heap);merged.append(value)
            if j+1<len(row):heapq.heappush(heap,(sums[i]+row[j+1],i,j+1))
        sums=merged
    return sums[k-1]

def _variant_jobs(instance):
    jobs=sorted(instance['jobs'],key=lambda job:(job[1],job[0],job[2]))
    limit=instance['max_jobs'];cooldown=instance['cooldown']
    if limit<0 or cooldown<0 or any(start>=end for start,end,value in jobs):
        raise ValueError('Invalid job limit, cooldown or interval')
    return jobs,min(limit,len(jobs)),cooldown

def solve_unseen_variant(instance):
    """Choose at most K nonoverlapping jobs, with a fixed gap after each selected job."""
    jobs,limit,cooldown=_variant_jobs(instance)
    ends=[end for start,end,value in jobs]
    predecessors=[bisect_right(ends,start-cooldown,hi=i) for i,(start,end,value) in enumerate(jobs)]
    previous=[0]*(len(jobs)+1)
    for _ in range(limit):
        current=[0]*(len(jobs)+1)
        for i,(start,end,value) in enumerate(jobs,1):
            current[i]=max(current[i-1],previous[predecessors[i-1]]+value)
        previous=current
    return previous[-1]

def reference_unseen_variant(instance):
    jobs,limit,cooldown=_variant_jobs(instance)
    best=0
    for count in range(limit+1):
        for subset in combinations(jobs,count):
            ordered=sorted(subset,key=lambda job:job[0])
            if all(a[1]+cooldown<=b[0] for a,b in zip(ordered,ordered[1:])):
                best=max(best,sum(job[2] for job in ordered))
    return best

## 正确性、前提与复杂度

带权调度按结束时间排序后，包含最后一份工作的解，其余工作全部属于已兼容前缀；选/不选覆盖全部解。新增计数层保持同一分解。行和合并中，已有和排在第 k 名之后的前缀不可能进入下一轮前 k：至少已有 k 个不大的前缀能配相同最小后缀；重数保留使此论证对相等值成立。

**代价：** 带权调度 O(n log n) 时间、O(n) 空间。迁移版 O(n log n+nK)，滚动两层 O(n) 空间。r 行的 Top-k 合并至多 O(rk log k)，工作空间 O(k)，另有读取与排序性验证成本。参考子集枚举指数级，仅限小数据。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

example={'jobs':[(1,3,20),(3,5,25),(4,6,30),(7,8,15)],'max_jobs':2,'cooldown':1}
show_table(['candidate','value','purpose'],[
 ['subset enumeration',reference_unseen_variant(example),'small independent reference'],
 ['predecessor + K-layer DP',solve_unseen_variant(example),'optimized transfer'],
 ['row-sum heap',kth_smallest_row_sum([[1,3,11],[2,4,6]],5),'duplicates count separately']])

candidate,value,purpose
subset enumeration,50,small independent reference
predecessor + K-layer DP,50,optimized transfer
row-sum heap,7,duplicates count separately


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from itertools import product
from random import Random
assert job_scheduling([1,2,3,3],[3,4,5,6],[50,10,40,70])==120
assert job_scheduling([],[],[])==0
assert job_scheduling([1,2],[2,3],[-1,-2])==0
assert kth_smallest_row_sum([[1,3,11],[2,4,6]],5)==7
assert kth_smallest_row_sum([[1,1],[2,2]],4)==3
rng=Random(17501)
for _ in range(160):
    mat=[sorted(rng.randrange(-3,8) for _ in range(rng.randrange(1,5))) for _ in range(rng.randrange(1,5))]
    expected=sorted(sum(choice) for choice in product(*mat))
    for k in sorted({1,len(expected),rng.randrange(1,len(expected)+1)}):
        assert kth_smallest_row_sum(mat,k)==expected[k-1]
# Different seed and fresh cases from the displayed teaching example.
heldout=Random(17599)
for _ in range(300):
    jobs=[]
    for i in range(heldout.randrange(10)):
        start=heldout.randrange(12);end=start+heldout.randrange(1,5)
        jobs.append((start,end,heldout.randrange(-4,21)))
    instance={'jobs':jobs,'max_jobs':heldout.randrange(6),'cooldown':heldout.randrange(4)}
    assert solve_unseen_variant(instance)==reference_unseen_variant(instance)
    ordinary={'jobs':jobs,'max_jobs':len(jobs),'cooldown':0}
    assert job_scheduling([x[0] for x in jobs],[x[1] for x in jobs],[x[2] for x in jobs])==reference_unseen_variant(ordinary)
large={'jobs':[(3*i,3*i+1,1) for i in range(5000)],'max_jobs':12,'cooldown':1}
assert solve_unseen_variant(large)==12
# No exponential reference on the large case.
assert kth_smallest_row_sum([[i for i in range(50)] for _ in range(10)],20)>=0
try:kth_smallest_row_sum([[1],[2]],2)
except ValueError:pass
else:raise AssertionError('Invalid rank was accepted')
print('N175: 所有本章断言通过')

N175: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用自编未展示答案的变体作正式考核，以上题只作热身。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 改变在线/加权/重复/规模中的一个条件并重新推导。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 42. Trapping Rain Water（comparison）
- 1235. Maximum Profit in Job Scheduling（comparison）
- 1439. Find the Kth Smallest Sum of a Matrix With Sorted Rows（comparison）：[官方入口](https://leetcode.com/problems/find-the-kth-smallest-sum-of-a-matrix-with-sorted-rows/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。